# Session 4 — Exercises

**Before session 5.**

Same house pattern: **predict, then measure.** Write down what you expect before you run anything.

Answers go in the **✍️ Your answer** cells. Code TODOs are marked `# TODO`.

Solutions are released with session 5, after these have been reviewed in class.

## Setup — given, just run it

In [ ]:
import sys
import urllib.request
from collections import Counter
from pathlib import Path

from langchain.chat_models import init_chat_model

HERE = Path.cwd()
sys.path.insert(0, str(HERE / "project"))
from config import settings                      # noqa: E402  — also loads .env
import triage as project                         # noqa: E402  — the SHIPPED triage(), not a copy


def ollama_running(url: str = "http://localhost:11434/api/tags", timeout: float = 2.0) -> bool:
    try:
        urllib.request.urlopen(url, timeout=timeout)
        return True
    except Exception:
        return False


TICKETS = project.load_tickets()
by_id = {t["id"]: t for t in TICKETS}

print(f"{len(TICKETS)} tickets · default {settings.model_id}")
print(f"ollama running: {ollama_running()}")

## ⭐ Exercise 1 — The swap

**What this is really teaching:** that *"we can switch providers"* is a claim about your **code**, not
about your **product**. The code swaps in one string — you watched it happen in section 5. The
behaviour does not swap. It changes, and somebody has to decide whether the change is acceptable
before it reaches a customer.

Run the project's own `triage()` — the shipped function in `project/triage.py`, not a reimplementation
— over the same tickets on **two** different models, and find one behavioural difference.

The richest free comparison tonight is the course default against **`ollama:llama3.2:3b`**, because
one of them is roughly a hundredth the size of the other and you can watch exactly what that buys.
Section 5.5 has the two-command setup. If you have a Groq key instead, `groq:openai/gpt-oss-20b`
works the same way.

Three rules, which are what make this an experiment rather than an anecdote:

- **Change one thing.** Same tickets, same prompt, same schema. Only the provider string moves.
- **Run each combination at least three times.** A difference you have seen once is noise. Session 3,
  appendix G, is a whole essay on people forgetting this.
- **Compare fields, not vibes.** `category`, `urgency`, `sentiment`, `needs_human` — four columns you
  can line up. "It felt worse" is not a finding.

Start with `TCK-001`, `TCK-003` and `TCK-009`: one straightforward, one that session 3 argues about,
and one where money has moved.

**✍️ Predict first:** which of the four fields do you expect to go wrong first on the smaller model —
and, harder, would anything in your program *notice* that it had?

**✍️ Your prediction:**

*(before you run anything)*

In [ ]:
CHALLENGER = "ollama:llama3.2:3b"      # or "groq:openai/gpt-oss-20b" if that is the key you have
IDS = ["TCK-001", "TCK-003", "TCK-009"]

# TODO 1: build two models with init_chat_model — one on settings.model_id, one on CHALLENGER.
#         Give the local one a generous timeout; it is slower than the hosted ones.

# TODO 2: for each ticket, call project.triage(model, ticket) THREE times on each model and print
#         category / urgency / needs_human for every run, side by side.

# TODO 3: find a field where they disagree in ALL THREE runs. That is the finding.
#         A field that disagrees once is the noise you already knew was there.

**✍️ Your answer.** Name the field, name the ticket, and give the three-run evidence. Then the part
that matters more than the observation:

- **What does that difference cost ShopWise?** Say it in terms of what the field is *for* — look at
  `schemas.py` and read what each field's `description` says it decides.
- **Would anything in your program have noticed?** The object validated. Session 3's word for this
  was *legal, not right* — which of your defences would have caught it, and which would not?
- One field probably came out **the same on both models even though the models disagreed about the
  ticket**. Find it, work out why, and say what that tells you about where to put a rule you actually
  care about.
- Finally, the engineering call: name one task you *would* run on the small local model, and one you
  would not. "It is worse" is not an answer — free, private and offline are real advantages, so say
  what they are worth and where they run out.

*(your answer here)*

## ⭐⭐ Exercise 2 — Trace archaeology

**What this is really teaching:** a trace is a *source*, not a dashboard. In a real incident nobody
hands you a summary — you have a project full of runs and a question, and the skill is turning the
question into a query. Tonight the question is small enough that you can check your own answer.

Run the three-ticket loop in the next cell. Then answer three questions **from the traces only**:

1. **How many tokens did the run cost?** Input and output, across all three calls.
2. **Which call was the slowest**, and by how much?
3. **Which ticket produced `needs_human=True`?**

The rule is the whole exercise: **you may not read the answer off what `triage()` returned, and you
may not add a `print`.** Pretend the run happened on Saturday and it is now Tuesday.

Notice that two of those are no longer answerable any other way. v0.2's `triage()` returned
`(TriageResult, usage_metadata)`; v1's returns the `TriageResult` alone, because in section 8 the
program stopped counting tokens. Nothing timed the calls either. So questions 1 and 2 exist *only*
in the trace now — which is the deal you accepted when `cost()` was deleted.

Question 3 looks like the easy one. It is not, and finding out why is the point of the exercise.

**✍️ Predict first:** which of the three calls do you expect to be slowest, and why? And which
ticket do you expect to come back `needs_human=True`? Write both down before you query anything.

In [ ]:
# Given — run this, then leave it alone. It makes the traces you are going to interrogate.
from datetime import datetime, timezone

from langchain_core.tracers.langchain import wait_for_all_tracers

IDS = ["TCK-001", "TCK-006", "TCK-009"]      # order status · a warranty question · a double charge

model = init_chat_model(settings.model_id, timeout=settings.REQUEST_TIMEOUT_S,
                        max_retries=settings.MAX_RETRIES)

RUN_STARTED = datetime.now(timezone.utc)
for tid in IDS:
    project.triage(model, by_id[tid])        # the return value is dropped on purpose
wait_for_all_tracers()                       # the tracer posts in a background thread; let it finish

print(f"three tickets triaged, starting {RUN_STARTED:%H:%M:%S} UTC. Now go and find them.")

In [ ]:
# TODO 1: connect and query.
#   - `Client()` from `langsmith`, then `client.read_project(project_name=...)` for the project id.
#   - `await client.runs.query(project_ids=[...], run_type="llm", min_start_time=RUN_STARTED, ...)`
#     is the current API. Two things catch everyone: it wants project *ids*, not names, and
#     `selects` is not optional — leave it out and every run comes back as a bare id.
#   - Runs are posted by a background thread and priced when the platform ingests them, so a
#     query fired immediately finds them with empty token counts. Flush with
#     `wait_for_all_tracers()` (from langchain_core.tracers.langchain), then poll until
#     `total_tokens` is set. The shape, once, so you are not guessing at the arguments:
#
#         client = Client()
#         pid = client.read_project(project_name=settings.LANGSMITH_PROJECT).id
#         page = await client.runs.query(
#             project_ids=[str(pid)], run_type="llm", min_start_time=RUN_STARTED, page_size=20,
#             selects=["ID", "NAME", "START_TIME", "LATENCY_SECONDS", "PROMPT_TOKENS",
#                      "COMPLETION_TOKENS", "TOTAL_TOKENS", "INPUTS_PREVIEW", "OUTPUTS_PREVIEW"])
#         runs = [r async for r in page]
#
#     Every one of these questions is also two clicks in the run list, which is what you would
#     do at work. Write the query anyway: once, so you have seen the shape.

# TODO 2: total tokens for the run. There are three token fields on a run and they do not all
#         mean the same thing — say in your answer which one you used and why.

# TODO 3: the slowest call. Print all three latencies, not just the winner: a maximum without the
#         other two is not evidence.

# TODO 4: which ticket produced `needs_human=True`. You will need the input side of the run to tell
#         the three apart, and the output side to read the answer.

# TODO 5: now ask the same question again, but of the ROOT run of each trace instead of the llm
#         run — pass `is_root=True` and drop `run_type`. Section 8.3 says what the difference
#         between those two rows is. If the two answers disagree, that disagreement is the most
#         interesting thing in this exercise and your answer has to explain it.

**✍️ Your answer.** Three numbers, then the part that matters more:

- **Total tokens**, and which field you read. Then: is that number the *cost* of the run? Check it
  against `data/prices.yml` and say what the run actually cost.
- **The slowest call**, with all three latencies. Then the harder question: is it slow *for a
  reason*, or is it one sample? What would you have to do before putting that number in a ticket?
- **Which ticket came back `needs_human=True`** — and whether the `llm` runs and the root runs agree
  with each other. If they do not, name the exact line of ShopWise's code that explains the
  difference, and say which of the two answers you would quote to a manager asking *"did the
  assistant escalate that ticket?"*

Finally, one sentence on the habit: you have just answered three questions about a run that had
finished before you asked. Name one thing you did last week with a `print` that this would have
answered faster.

## ⭐⭐ Exercise 3 — Ship a prompt change

**What this is really teaching:** the loop end to end, on a change *you* invented. Section 9 walked
it with a candidate that was already written and already scored, which is the easy version. Nobody
has run yours, and the number is the only thing that gets to decide.

Start by reading the misses in section 9's A/B output. The shipped wording gets two of the twenty
wrong, and they are the same two most evenings. **A change you cannot connect to a specific miss is a
guess with a version number on it.**

Then, in order:

1. **Write one candidate wording.** One change, for a reason you can say in a sentence. Give it a
   name you would say out loud — `v2-something`, not `test2`.
2. **Score both arms** over all twenty tickets — the shipped wording and yours, same tickets, same
   model, same schema.
3. **Find both arms in the trace**, by the tag the harness stamped on them. Report each arm's
   `prompt_version`, its `prompt_sha`, its call count and what it spent.
4. **Decide.** Ship it or do not, from the number. If you would ship it, say exactly which files
   change and why they have to change in the *same* commit.

**One rule, and it is the whole exercise: change one thing.** If you improve the wording *and*
reorder it *and* add an example, the result is uninterpretable — the number moved and you cannot say
which of the three did it. That is not a smaller version of a good experiment; it is not an
experiment.

**✍️ Predict first:** which miss do you expect your change to fix, and — the half people skip —
what do you expect it to **break**? Write both down before you run anything.

In [ ]:
# TODO 1: your candidate wording.
#   Copy `triage_prompt` out of project/prompts.yml and change ONE thing. Do not edit prompts.yml
#   itself yet: the application ships one wording, and a candidate is not a wording you ship — it
#   is one you are still arguing about. That is why score_triage.py holds PROMPT_CANDIDATES.
CANDIDATE_NAME = "v2-"          # a label a human would say out loud
CANDIDATE = """
"""

# TODO 2: score both arms over all twenty tickets.
#   score_triage.py sits beside this notebook, so import it rather than reimplementing it:
#
#       import score_triage as harness
#
#       base = harness.score(model, TICKETS, label="shipped", slug=project.prompt_version())
#       cand = harness.score(model, TICKETS, label=CANDIDATE_NAME, slug=CANDIDATE_NAME,
#                            prompt=CANDIDATE, version=CANDIDATE_NAME)
#
#   Each returns a Run carrying .hits, .misses, .out_tokens, .spent and .tag; .report() prints it,
#   and harness.compare(base, cand, bought="...") prints the verdict. You will need a `model` —
#   init_chat_model, exactly as exercise 2 built one.
#   That is 40 live calls and a couple of minutes. It is the price of knowing.

# TODO 3: find both arms in the trace.
#   Every call the harness made carries `harness.RUN_TAG`, and each arm additionally carries
#   f"{harness.RUN_TAG}-{slug}". Query with `filter='has(tags, "...")'`, and do not forget
#   run_type="llm" — section 9.3 of the lesson notebook says what happens if you do.
#   Print, per arm: prompt_version, prompt_sha, calls, output tokens, cost.

# TODO 4: the decision.
#   Which tickets moved and in which direction? Is the difference bigger than the noise on a
#   20-ticket set? Then the question a reviewer actually asks: would you deploy this on Monday?

**✍️ Your answer.**

- **The change, in one sentence** — what you altered and which miss it was aimed at.
- **The two numbers**, both arms, out of twenty. Then the tickets that moved, in both directions.
- **The two runs in the trace** — both arms' `prompt_version` and `prompt_sha`. Then a question
  worth more than it looks: if the two arms had come back with the *same* hash, what would have gone
  wrong, and where?
- **Ship or not, and why.** If not, say what you would try next and what evidence would change your
  mind. If yes, name the files that change together in one commit, and say how you would prove after
  the deploy that the new wording is actually the one production is running.

And one sentence on the thing this exercise cannot teach you in twenty tickets: what would you need
before you trusted a **one-ticket** difference?

## ⭐⭐⭐ Exercise 4 — The same request, three bills

**What this is really teaching:** what a standard interface does *not* standardise. Your code will
be identical for all three providers. The bill will not be, and it will not be close. This is the
exercise that stops you saying *"we can swap providers, it is one string"* in a design review without
adding the second half of the sentence.

There is also a specific API trap in the middle of it, and it is the one everybody hits exactly once
— usually in production, on the day somebody asks what the feature costs.

**The task.** Take **one** ticket. Run it through **three** provider strings, using
`with_structured_output(TriageResult, include_raw=True)`. For each provider, report:

1. the category it came back with;
2. the **input tokens** the call was billed;
3. the same request's input tokens **with no schema attached at all**;
4. the difference between those two — the schema's price on that provider;
5. how the answer arrived: as JSON in the message text, or as a forced tool call.

Then answer the question the table poses: **why does one identical request bill one provider
hundreds of input tokens more than another, on every call, forever?**

**Two rules.**

- **Three providers you can actually run.** Gemini and Groq are free tiers; Ollama is local and needs
  no key. Section 1.1 of the lesson notebook lists what is available on your machine. Three is the
  minimum for a *pattern*; two is a comparison and one is an anecdote.
- **Same ticket, same prompt, same schema, every time.** The only variable is the provider string.
  Use the shipped prompt — `project.build_triage_prompt()` — rather than writing a new one, or you
  are measuring your prompt instead of the providers.

**The trap, stated plainly so that you notice when you hit it:** `usage_metadata` is **not** on the
parsed object. `out["parsed"]` is your `TriageResult` and a `TriageResult` has no tokens, no latency
and no id — it is your data class, and nothing else survived. The numbers are on `out["raw"]`, which
is the only reason `include_raw=True` exists. Write down what the error message says when you try it
the other way; that error message is what you will actually see at 11pm.

**✍️ Predict first**, before you run anything:

- Which of your three providers do you expect to bill the **most** for the schema, and why?
- Do you expect a schema to cost input tokens **at all**? Say what you think is physically happening
  to `TriageResult` between your Python process and the model.
- Will all three agree on the category?

In [ ]:
# TODO 1: pick three provider strings you can run, and one ticket. Any ticket — but the SAME
#         ticket for all three. `by_id["TCK-001"]` is a reasonable choice.

# TODO 2: for each provider, make TWO calls with identical messages:
#           a) with_structured_output(TriageResult, include_raw=True).invoke(...)   -> the schema
#           b) plain .invoke(...)                                                    -> no schema
#         The second one is what isolates the schema's cost. Without it you are comparing
#         providers' baseline verbosity and calling it a schema bill.

# TODO 3: pull usage_metadata off the RAW message: out["raw"].usage_metadata.
#         First, deliberately try out["parsed"].usage_metadata and record the exact error.

# TODO 4: work out how the answer arrived. `out["raw"].tool_calls` is non-empty when the provider
#         had no JSON mode and LangChain fell back to forcing a tool call. When it is empty, the
#         answer came back as text. Print which, per provider.

# TODO 5: one table. provider · category · input tokens without schema · with schema · the
#         difference · the mechanism. Sort it by the difference, because the ordering is the point.

# TODO 6: scale it. At 1,000 tickets a day, how many extra INPUT tokens does the most expensive
#         provider burn against the cheapest — on the schema alone? Then find one of these calls
#         in LangSmith and read the dollars off the trace rather than computing them.

**✍️ Your answer.**

- **The table**, all three providers, with the difference column sorted.
- **The trap**: the exact `AttributeError` you got from the parsed object, and one sentence on why
  `include_raw=True` exists at all — what is it giving back that the short form throws away?
- **The explanation.** Why is the spread this large? Your answer has to connect the *mechanism*
  column to the *cost* column, and it has to survive this objection: *"the expensive one is just a
  chattier model"* — which your no-schema numbers can refute or confirm. Say which.
- **The two axes.** Is "sends the schema in the request" the same thing as "uses a tool call"? Your
  table either shows those two columns agreeing or it does not. Say what that means about what
  `with_structured_output` is actually doing on each provider.
- **The money**, at 1,000 tickets a day.
- **The decision.** You are asked to move ShopWise to the most expensive provider in your table
  because someone benchmarked it as smarter. What do you say, in one sentence, with a number in it?
  And what would you do to the schema first?